### Table Name : _**crm_sales_details**_

In [0]:
from pyspark.sql.functions import *

df = spark.read.table('data_lakehouse.bronze.crm_sales_details')

df.display()

### Triming the extra space from string columns

In [0]:
for fields in df.schema:
    if isinstance(fields.dataType,StringType):
        df = df.withColumn(fields.name,trim(fields.name))

df.display()

### Updating Date format & invalid dates

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = (
    df.withColumn('sls_order_dt',when((col('sls_order_dt') == 0) | (length(col('sls_order_dt')) != 8), None)
                   .otherwise(date_format(to_date(col("sls_order_dt").cast('string'),'yyyyMMdd'),'yyyy/MM/dd')))
    .withColumn('sls_ship_dt',when((col('sls_ship_dt') == 0) | (length(col('sls_ship_dt')) != 8), None)
                   .otherwise(date_format(to_date(col("sls_ship_dt").cast('string'),'yyyyMMdd'),'yyyy/MM/dd')))
    .withColumn('sls_due_dt',when((col('sls_due_dt') == 0) | (length(col('sls_due_dt')) != 8), None)
                   .otherwise(date_format(to_date(col("sls_due_dt").cast('string'),'yyyyMMdd'),'yyyy/MM/dd')))
)

df.display()

In [0]:
df = df.withColumn('sls_price',when(col("sls_price").isNull() | (col("sls_price") != col("sls_sales")/col('sls_quantity')), 
                    (col("sls_sales")/col('sls_quantity')))
                   .otherwise(col("sls_price"))
                   )

df.display()

### Column Renaming

In [0]:
RENAMED_COLUMN = {
    
    'sls_ord_num' : 'sales_order_number',
    'sls_prd_key' : 'sales_product_key',
    'sls_cust_id' : 'sales_customer_id',
    'sls_order_dt' : 'sales_order_date',
    'sls_ship_dt' : 'sales_ship_date',
    'sls_due_dt' : 'sales_due_date',
    'sls_sales' : 'sales_amount',
    'sls_quantity' : 'sales_quantity',
    'sls_price' : 'sales_price'

}

In [0]:
for old_name,new_name in RENAMED_COLUMN.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('data_lakehouse.silver.crm_sales_info')

In [0]:
%sql
select * from data_lakehouse.silver.crm_sales_info